In [ ]:
# Clean install to avoid NumPy ABI issues (run after runtime restart)
!apt-get -qq update
!apt-get -qq install -y ffmpeg




!pip install -q --upgrade pip

!pip uninstall -y numpy pandas scipy scikit-learn pyarrow
!pip install -U --no-cache-dir --force-reinstall numpy pandas scipy scikit-learn pyarrow
!pip install -U --no-cache-dir transformers datasets accelerate soundfile



In [ ]:
!pip install transformers torch datasets tensorboard soundfile librosa

**Reasoning**:
To allow the user to easily update the `data_folder_path` and ensure it's explicitly defined, I will provide a code cell where this variable is set. This fulfills the first instruction of reviewing the variable and allows for direct modification.



**Reasoning**:
Now that the `data_folder_path` variable has been defined, I will execute the provided Python code to verify its existence, list its contents, and confirm the presence of the 'record' and 'transcriptions' subfolders. This addresses the second instruction of the subtask.



# Setting basic paths

In [ ]:
import os, json, pathlib, subprocess, hashlib, wave, threading
import numpy as np
from tqdm.auto import tqdm
from transformers import WhisperProcessor, WhisperForConditionalGeneration

# ----------------------------
# USER SETTINGS
# ----------------------------
ACFT_MODEL_ID = "futo-org/acft-whisper-tiny.en"     # weights
BASE_PROCESSOR_ID = "openai/whisper-tiny.en"        # processor (feature extractor + tokenizer)

TRANSCRIPT_DIR = "/content/drive/MyDrive/Transcriptions"
CHUNKS_DIR = "/content/drive/MyDrive/Record_chunks"

TARGET_SR = 16000
MAX_OUT_SECONDS = 30.0
MAX_OUT_FRAMES = int(MAX_OUT_SECONDS * TARGET_SR)
DUR_CAP_SEC = (MAX_OUT_FRAMES - 1) / float(TARGET_SR)  # ~29.9999s at 16k

# Training-label safety
MAX_LABEL_TOKENS = 420

# Segment-level behaviour
CONTEXT_PAD = 0.10            # small padding (0.05–0.20 recommended)
MIN_SEG_SEC = 0.80            # if a segment is shorter than this, merge forward
MERGE_GAP_FOR_SHORT = 0.25    # only merge short segments if the gap is <= this
KEEP_TINY_SEGMENTS = False    # if still < MIN_SEG_SEC after merging, keep anyway?

# Groq no_speech_prob is often not reliable for filtering. Leave disabled.
# If YOUR metadata is reliable, set something like 0.95.
NO_SPEECH_PROB_DROP = None

# Output files
CHUNKS_DIR_P = pathlib.Path(CHUNKS_DIR)
CHUNKS_DIR_P.mkdir(parents=True, exist_ok=True)

MANIFEST_PATH = str(CHUNKS_DIR_P / "pairs_manifest.jsonl")
PENDING_PAIRS_PATH = str(CHUNKS_DIR_P / "pairs_pending.jsonl")
PENDING_TASKS_PATH = str(CHUNKS_DIR_P / "tasks_pending.jsonl")

# Use the *base* processor (has preprocessor_config.json etc.)
processor = WhisperProcessor.from_pretrained(BASE_PROCESSOR_ID)

# If you actually want to run inference with the ACFT weights:
model = WhisperForConditionalGeneration.from_pretrained(ACFT_MODEL_ID)






# ----------------------------
# MANIFEST HELPERS
# ----------------------------

def load_processed_jsons_from_manifest(path: str) -> set:
    processed = set()
    if not os.path.exists(path):
        return processed
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                tj = obj.get("transcript_json")
                if tj:
                    processed.add(tj)
            except json.JSONDecodeError:
                continue
    return processed


def write_jsonl_overwrite(path: str, rows: list) -> None:
    pathlib.Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for obj in rows:
            f.write(json.dumps(obj, ensure_ascii=False) + "\n")


def append_manifest_jsonl(path: str, rows: list) -> None:
    if not rows:
        return
    pathlib.Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8") as f:
        for obj in rows:
            f.write(json.dumps(obj, ensure_ascii=False) + "\n")


def read_jsonl(path: str) -> list:
    rows = []
    if not os.path.exists(path):
        return rows
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows






# ----------------------------
# FAST HELPERS
# ----------------------------

_ffprobe_cache = {}
_ffprobe_lock = threading.Lock()

def ffprobe_duration_sec(path: str) -> float:
    with _ffprobe_lock:
        if path in _ffprobe_cache:
            return _ffprobe_cache[path]
    cmd = [
        "ffprobe", "-v", "error",
        "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1",
        path,
    ]
    out = subprocess.check_output(cmd).decode("utf-8").strip()
    dur = float(out)
    with _ffprobe_lock:
        _ffprobe_cache[path] = dur
    return dur


def safe_id_from_path(p: str, n: int = 10) -> str:
    return hashlib.sha1(p.encode("utf-8")).hexdigest()[:n]


def chunk_filename(audio_path: str, transcript_json_path: str, chunk_index: int) -> str:
    base = pathlib.Path(audio_path).stem
    jid = safe_id_from_path(transcript_json_path, n=10)
    return f"{base}__{jid}_chunk{chunk_index:04d}.wav"


def is_useless_segment(seg: dict) -> bool:
    txt = (seg.get("text") or "").strip()
    if txt in {"", ".", "…"}:
        return True

    if NO_SPEECH_PROB_DROP is not None:
        nsp = seg.get("no_speech_prob", None)
        if nsp is not None and float(nsp) >= float(NO_SPEECH_PROB_DROP):
            return True

    s = seg.get("start", None)
    e = seg.get("end", None)
    if s is None or e is None:
        return True
    if float(e) <= float(s):
        return True

    return False


# NEW FUNCTION: resolve_audio_path
def resolve_audio_path(audio_path_from_json: str) -> str | None:
    # Try the path exactly as it is given
    if os.path.exists(audio_path_from_json):
        return audio_path_from_json

    dir_name = os.path.dirname(audio_path_from_json)
    base_name_stem = os.path.splitext(os.path.basename(audio_path_from_json))[0]

    # If the directory doesn't exist, we can't resolve anything
    if not os.path.exists(dir_name):
        return None

    # Common audio extensions to check (case-insensitive)
    possible_extensions = ['.m4a', '.wav', '.mp3', '.flac', '.ogg']

    # Iterate through files in the directory to find a case-insensitive match for the base name
    # and a matching audio extension.
    for fname in os.listdir(dir_name):
        fname_stem, fname_ext = os.path.splitext(fname)
        if fname_stem.lower() == base_name_stem.lower():
            if fname_ext.lower() in possible_extensions:
                full_resolved_path = os.path.join(dir_name, fname)
                # Final check to ensure the resolved path actually exists
                if os.path.exists(full_resolved_path):
                    return full_resolved_path
    return None


# Core must fit into a <=30s window after padding.
MAX_CORE_ALLOWED = MAX_OUT_SECONDS - 2.0 * CONTEXT_PAD


def _estimate_segment_tokens(segments: list) -> list:
    """Token length estimate using WhisperProcessor tokenizer.

    IMPORTANT: This uses seg['text'] only.
    We do NOT use seg['tokens'] from Groq.
    """
    n = len(segments)
    seg_tok = [0] * n

    useful_idx, useful_txt = [], []
    for i, seg in enumerate(segments):
        if is_useless_segment(seg):
            continue
        t = (seg.get("text") or "").strip().lower()
        if t:
            useful_idx.append(i)
            useful_txt.append(t)

    if useful_txt:
        enc = processor.tokenizer(useful_txt, add_special_tokens=False)
        lens = [len(ids) for ids in enc["input_ids"]]
        for i, L in zip(useful_idx, lens):
            seg_tok[i] = int(L)

    return seg_tok


def build_segment_level_chunks(segments: list) -> list:
    """Segment-level chunks.

    Rule:
    - Each chunk starts from one segment.
    - If that segment is shorter than MIN_SEG_SEC, merge forward while:
        * next segment is not useless
        * gap <= MERGE_GAP_FOR_SHORT
        * label tokens stay <= MAX_LABEL_TOKENS
        * core length stays <= MAX_CORE_ALLOWED
    - target_out_sec = core_span + 2*CONTEXT_PAD (capped to MAX_OUT_SECONDS)
    """

    n = len(segments)
    seg_tok = _estimate_segment_tokens(segments)

    chunks = []
    i = 0

    while i < n:
        seg = segments[i]
        if is_useless_segment(seg):
            i += 1
            continue

        s = float(seg["start"])
        e = float(seg["end"])
        txt = (seg.get("text") or "").strip()
        if not txt:
            i += 1
            continue

        cur_s, cur_e = s, e
        texts = [txt]
        tok_sum = seg_tok[i]

        # If too short, merge forward cautiously.
        while (cur_e - cur_s) < MIN_SEG_SEC and (i + 1) < n:
            nxt = segments[i + 1]
            if is_useless_segment(nxt):
                i += 1
                continue

            ns = float(nxt["start"])
            ne = float(nxt["end"])
            gap = ns - cur_e
            if gap > MERGE_GAP_FOR_SHORT:
                break

            ntext = (nxt.get("text") or "").strip()
            if not ntext:
                i += 1
                continue

            # Safety: token cap
            ntoks = seg_tok[i + 1]
            if (tok_sum + ntoks) > MAX_LABEL_TOKENS:
                break

            # Safety: core must fit
            if (ne - cur_s) > MAX_CORE_ALLOWED:
                break

            # Accept merge
            texts.append(ntext)
            tok_sum += ntoks
            cur_e = ne
            i += 1

        core_span = cur_e - cur_s
        if core_span < MIN_SEG_SEC and not KEEP_TINY_SEGMENTS:
            i += 1
            continue

        target_out = float(min(MAX_OUT_SECONDS, core_span + 2.0 * CONTEXT_PAD))

        chunks.append({
            "start": float(cur_s),
            "end": float(cur_e),
            "text": " ".join([t.strip() for t in texts]).strip(),
            "target_out_sec": target_out,
        })

        i += 1

    return chunks


def compute_centered_window(audio_total: float, core_start: float, core_end: float, target_out: float):
    core_start = float(max(0.0, core_start))
    core_end = float(min(audio_total, core_end))
    if core_end < core_start:
        core_end = core_start

    core_span = core_end - core_start
    target_out = float(min(target_out, MAX_OUT_SECONDS))

    # Must contain core + context padding if possible
    need = min(MAX_OUT_SECONDS, max(target_out, core_span + 2.0 * CONTEXT_PAD))

    mid = 0.5 * (core_start + core_end)
    start = mid - need / 2.0
    start = max(0.0, min(start, audio_total - need))
    return start, need






# ----------------------------
# MAIN: PLAN + SAVE PENDING JSONL
# ----------------------------

TRANSCRIPT_DIR_P = pathlib.Path(TRANSCRIPT_DIR)
json_files = sorted(TRANSCRIPT_DIR_P.glob("*.json"))

processed_jsons = load_processed_jsons_from_manifest(MANIFEST_PATH)

# Load transcripts that were part of a previous pending plan
pending_processed_jsons_from_file = set()
if os.path.exists(PENDING_PAIRS_PATH):
    for obj in read_jsonl(PENDING_PAIRS_PATH):
        if "transcript_json" in obj:
            pending_processed_jsons_from_file.add(obj["transcript_json"])

# Combine all processed/pending transcripts
all_already_processed_jsons = processed_jsons.union(pending_processed_jsons_from_file)

new_json_files = [p for p in json_files if str(p) not in all_already_processed_jsons]

print("Found JSONs:", len(json_files))
print("Already processed (final manifest):", len(processed_jsons))
print("Already processed (pending file):", len(pending_processed_jsons_from_file))
print("Planning NEW JSONs now:", len(new_json_files))

pending_pairs = []
pending_tasks = []
bad_json = 0
bad_audio = 0

for jf in tqdm(new_json_files, desc="Planning segment-level chunks"):
    jf_str = str(jf)

    try:
        obj = json.loads(jf.read_text(encoding="utf-8"))
        audio_path_from_json = obj["input_file"]["path"]
        segments = obj["groq_response"]["segments"]
    except Exception:
        bad_json += 1
        continue

    # Resolve the actual audio path, considering different extensions
    resolved_audio_path = resolve_audio_path(audio_path_from_json)

    if resolved_audio_path is None:
        bad_audio += 1
        continue

    chunks = build_segment_level_chunks(segments)
    if not chunks:
        continue

    for idx, ch in enumerate(chunks):
        out_name = chunk_filename(resolved_audio_path, jf_str, idx)
        out_wav = str(CHUNKS_DIR_P / out_name)

        target_out = float(min(ch["target_out_sec"], MAX_OUT_SECONDS))

        pending_pairs.append({
            "audio_path": out_wav,
            "raw_transcription": ch["text"],     # <--- ALWAYS text, never Groq tokens
            "source_audio": resolved_audio_path,
            "chunk_index": idx,
            "transcript_json": jf_str,
            "duration_sec_target": target_out,
            "sr": TARGET_SR,
            "model": ACFT_MODEL_ID, # Changed from MODEL to ACFT_MODEL_ID
        })

        # Only cut if the wav doesn't already exist
        if not os.path.exists(out_wav):
            pending_tasks.append({
                "audio_path": resolved_audio_path,
                "out_wav": out_wav,
                "core_start": float(ch["start"]),
                "core_end": float(ch["end"]),
                "target_out_sec": target_out,
            })

write_jsonl_overwrite(PENDING_PAIRS_PATH, pending_pairs)
write_jsonl_overwrite(PENDING_TASKS_PATH, pending_tasks)

print("\nSaved pending files:")
print("  pending pairs:", PENDING_PAIRS_PATH, "| rows:", len(pending_pairs))
print("  pending tasks:", PENDING_TASKS_PATH, "| rows:", len(pending_tasks))
print("Bad JSON:", bad_json, "| Missing audio:", bad_audio)


# Cutting

In [1]:
import numpy as np
import sounddevice as sd

sampling_rate = 44100  # samples per second
duration = 0.5         # seconds
frequency = 440        # Hz (A4 note)

t = np.linspace(0, duration, int(sampling_rate * duration), False)
audio_data = 0.5 * np.sin(2 * np.pi * frequency * t).astype(np.float32)

sd.play(audio_data, samplerate=sampling_rate)
sd.wait()

ModuleNotFoundError: No module named 'sounddevice'

# analysis of distribution of duration of files in the record chunks

In [ ]:
# ========= TEXT-ONLY WAV DURATION DISTRIBUTION (FAST) =========
import os, math, wave, shutil, subprocess
import numpy as np
from concurrent.futures import ThreadPoolExecutor, as_completed
from multiprocessing import Pool

# ----------------------------
# SETTINGS
# ----------------------------
DRIVE_CHUNKS_DIR = "/content/drive/MyDrive/Record_chunks"

# BIG speed-up option: copy Drive folder to local VM disk first (recommended for many small files)
USE_LOCAL_COPY = False
LOCAL_CHUNKS_DIR = "/content/Record_chunks_local"   # will be created/overwritten by rsync copy

# Duration limits (for reporting)
MIN_SEC = 2.0
MAX_SEC = 30.0

# Text histogram
BIN_WIDTH = 0.25       # 0.25 = detailed, 0.5 = faster/cleaner output
HIST_MAX_SEC = 30.0    # histogram range (keep to 30 for your use-case)
BAR_WIDTH = 60         # characters
SHOW_ZERO_BINS = False # True if you want every bin printed

# Concurrency mode: "threads" (recommended) or "processes" or "single"
MODE = "threads"

# Workers / batching
WORKERS = min(32, (os.cpu_count() or 4) * 2)   # threads: okay to be higher
PROC_WORKERS = max(1, (os.cpu_count() or 2) - 1)
PROC_CHUNKSIZE = 4096                          # bigger reduces overhead for tiny tasks

# If you just want a quick sample run (set None for all)
MAX_FILES = None

# ----------------------------
# FAST recursive scandir
# ----------------------------
def iter_wavs(root):
    stack = [root]
    while stack:
        d = stack.pop()
        try:
            with os.scandir(d) as it:
                for e in it:
                    if e.is_dir(follow_symlinks=False):
                        stack.append(e.path)
                    elif e.is_file(follow_symlinks=False) and e.name.lower().endswith(".wav"):
                        yield e.path
        except (FileNotFoundError, PermissionError):
            continue

def wav_duration_sec(path):
    # Reads just the header; fast. wave supports uncompressed PCM WAV. :contentReference[oaicite:4]{index=4}
    try:
        with wave.open(path, "rb") as wf:
            fr = wf.getframerate()
            nf = wf.getnframes()
        if fr <= 0:
            return None
        return nf / float(fr)
    except Exception:
        return None

def ascii_hist(durs, bin_width=0.5, max_sec=30.0, bar_width=50, show_zero=False):
    durs = np.asarray(durs, dtype=np.float32)
    bins = np.arange(0.0, max_sec + bin_width, bin_width)
    counts, edges = np.histogram(durs, bins=bins)
    peak = int(counts.max()) if counts.size else 0

    print(f"\nASCII histogram (bin={bin_width}s)  n={len(durs)}")
    for i, c in enumerate(counts):
        if (not show_zero) and c == 0:
            continue
        lo, hi = edges[i], edges[i+1]
        bar_len = int((c / peak) * bar_width) if peak else 0
        bar = "█" * bar_len
        print(f"{lo:5.2f}-{hi:5.2f}s | {c:8d} {bar}")

def pct(durs, p):
    return float(np.percentile(durs, p)) if len(durs) else float("nan")

def maybe_copy_to_local(src_drive_dir, dst_local_dir):
    if not USE_LOCAL_COPY:
        return src_drive_dir

    # Fresh copy (delete old local dir if it exists)
    if os.path.exists(dst_local_dir):
        shutil.rmtree(dst_local_dir, ignore_errors=True)
    os.makedirs(dst_local_dir, exist_ok=True)

    # rsync is generally fast for directory trees (note: Drive mount itself can be slow for many small files). :contentReference[oaicite:5]{index=5}
    # Use -a to preserve structure. Avoid -z compression for many small files (often not helpful). :contentReference[oaicite:6]{index=6}
    cmd = ["rsync", "-a", src_drive_dir.rstrip("/") + "/", dst_local_dir.rstrip("/") + "/"]
    subprocess.run(cmd, check=True)
    return dst_local_dir

def collect_paths(root):
    paths = []
    for p in iter_wavs(root):
        paths.append(p)
        if MAX_FILES is not None and len(paths) >= MAX_FILES:
            break
    return paths

def scan_durations(paths):
    if MODE == "single":
        out = [wav_duration_sec(p) for p in paths]

    elif MODE == "threads":
        out = [None] * len(paths)
        with ThreadPoolExecutor(max_workers=WORKERS) as ex:
            futs = {ex.submit(wav_duration_sec, p): idx for idx, p in enumerate(paths)}
            for f in as_completed(futs):
                idx = futs[f]
                try:
                    out[idx] = f.result()
                except Exception:
                    out[idx] = None

    elif MODE == "processes":
        # Pool + large chunksize reduces IPC overhead for tiny tasks. :contentReference[oaicite:7]{index=7}
        with Pool(processes=PROC_WORKERS) as pool:
            out = list(pool.imap_unordered(wav_duration_sec, paths, chunksize=PROC_CHUNKSIZE))
    else:
        raise ValueError("MODE must be one of: threads, processes, single")

    durs = np.array([d for d in out if d is not None], dtype=np.float32)
    return durs

# ----------------------------
# RUN
# ----------------------------
if not os.path.exists(DRIVE_CHUNKS_DIR):
    raise SystemExit(f"Drive chunks dir not found: {DRIVE_CHUNKS_DIR}")

ROOT = maybe_copy_to_local(DRIVE_CHUNKS_DIR, LOCAL_CHUNKS_DIR)

paths = collect_paths(ROOT)
print(f"Mode={MODE} | root={ROOT}")
print(f"Found {len(paths):,} wav files" + (f" (sampled MAX_FILES={MAX_FILES})" if MAX_FILES else ""))

if not paths:
    raise SystemExit("No .wav files found. Check CHUNKS_DIR.")

durs = scan_durations(paths)
print(f"Readable durations: {len(durs):,} / {len(paths):,}")
if len(durs) == 0:
    raise SystemExit("Could not read durations from any wav. Files might be corrupt or not WAV PCM.")

outside = int(np.sum((durs < MIN_SEC) | (durs > MAX_SEC)))
print(
    f"\nDuration stats (sec):\n"
    f"  min={durs.min():.3f}\n"
    f"  p50={pct(durs,50):.3f}  mean={durs.mean():.3f}\n"
    f"  p90={pct(durs,90):.3f}  p95={pct(durs,95):.3f}  p99={pct(durs,99):.3f}\n"
    f"  max={durs.max():.3f}\n"
    f"  outside [{MIN_SEC}, {MAX_SEC}] sec: {outside:,} ({outside/len(durs)*100:.2f}%)"
)

# Text-based histogram up to 30s
ascii_hist(
    durs,
    bin_width=BIN_WIDTH,
    max_sec=HIST_MAX_SEC,
    bar_width=BAR_WIDTH,
    show_zero=SHOW_ZERO_BINS
)


# Just making noise

In [ ]:
import numpy as np
from IPython.display import Audio

sampling_rate = 44100  # samples per second
duration = 5.0        # seconds
frequency = 440       # Hz (A4 note)

t = np.linspace(0, duration, int(sampling_rate * duration), False)
audio_data = 0.5 * np.sin(2 * np.pi * frequency * t)

Audio(audio_data, rate=sampling_rate)